# Diabetes Classification (Similar to Iris Example)
This notebook demonstrates a simple machine learning workflow using the **Pima Indians Diabetes dataset**.
The structure is similar to a typical Iris classification example: load data, explore, split data, train models, and evaluate performance.

## 1. Import Libraries

In [1]:
# Load libraries
import os, sys #os dung de doc duong dan file, tao xoa thu muc kiem tra file ton tai,sys lay tham so dong lenh vaf dung thoat chuong trinh
from IPython import display # dung cho viec hien hinh anh
import numpy as np
import matplotlib.pyplot as plt # ve do thi 
import pandas as pd # dung su ly du lieu dang bang doc , loc du lieu tu csv excel
import seaborn as sns # ve bieu do nang cao
import joblib # luu va load mo hinh machine learning

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder # chuyen du lieu thanh vecto ,text thanh so, du lieu thu tu
from sklearn.preprocessing import MinMaxScaler, StandardScaler # chuan hoa du lieu tu 0 ->1,Chuẩn hóa dữ liệu theo mean = 0 và std = 1
from sklearn.model_selection import train_test_split # chia va train/ test

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams['figure.dpi'] = 100

warnings.filterwarnings("ignore")

## 2. Load Dataset

In [2]:
data_path = "data/pima-indians-diabetes.csv"

data_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df_dataset = pd.read_csv(data_path, names=data_names)

## 3. Phân tích dữ liệu (Analyze Data)

In [3]:
# shape
print(f'+ Shape : {df_dataset.shape}')
# types
print(f'+ Data Types: \n{df_dataset.dtypes}')
# head, tail
print(f'+ Contents: ')
display.display(df_dataset.head(5))
display.display(df_dataset.tail(5))
# info
df_dataset.info()

+ Shape : (768, 9)
+ Data Types: 
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object
+ Contents: 


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1
767,1,93,70,31,0,30.4,0.315,23,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
description = df_dataset.describe().T
display.display(description)

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


**Nhận xét**:
+ Dữ liệu có 9 tính chất để phân lớp: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, Outcome
+ Giá trị của từng tính chất lần lượt:số lần mang thai, mg/dL, mmHg, mm, µU/mL, kg/m², giá trị càng lờn càng bị
+ Tổng số dòng dữ liệu là 768 dòng
+ Dữ liệu để phân lớp ở cột Outcome

#### (2) **Kiểm tra tính toàn vẹn của dữ liệu**
+ Dữ liệu có bị trùng lặp không? Hiển thị dòng bị vi phạm.
+ Dữ liệu có tồn tại giá trị Null không? Hiển thị dòng bị vi phạm.
+ Dữ liệu có tồn tại giá trị NaN không? Hiển thị dòng bị vi phạm.

In [5]:
has_null = df_dataset.isnull().sum().any() # tim null, roi xem co bao nhieu gia tri null,check co cai nao lon hon 0 ko
has_nan  = df_dataset.isna().sum().any()
n_duplicated = df_dataset.duplicated().sum()
print(f'Tính toàn vẹn dữ liệu:')
print(f'+ Có giá trị Null: {has_null}')
if has_null:
    display.display(df_dataset[df_dataset.isnull().any(axis=1)])
print(f'+ Có giá trị Nan: {has_nan}')
if has_nan:
    display.display(df_dataset[df_dataset.isna().any(axis=1)])
print(f'+ Số dòng trùng: {n_duplicated}')


Tính toàn vẹn dữ liệu:
+ Có giá trị Null: False
+ Có giá trị Nan: False
+ Số dòng trùng: 0


#### (3) **Các tính chất thống kê trên dữ liệu số**


In [6]:
description = df_dataset.describe().T
display.display(description)

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


#### (4) **Tần số xuất hiện (Distribution) trên dữ liệu phân lớp (Outcome) và dữ liệu danh mục (Category)**


In [7]:
df_dataset["Outcome"].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

## Kiểm tra và loại bỏ dữ liệu lỗi, ngoại lệ

In [8]:
physiological_ranges = {
    "Pregnancies": (0, 20),
    "Glucose": (0, 200),
    "BloodPressure": (0, 140),
    "SkinThickness": (0, 100),
    "Insulin": (0, 900),
    "BMI": (0, 70),
    "DiabetesPedigreeFunction": (0, 2.5),
    "Age": (0, 120)
}

In [9]:
def detect_physiological_errors(df, physiological_ranges):
    errors = {}

    zero_invalid_cols = [
        "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"
    ]

    for col, (min_val, max_val) in physiological_ranges.items():
        if col not in df.columns:
            continue
        
        invalid_mask = (df[col] < min_val) | (df[col] > max_val)

        # chỉ thêm điều kiện == 0 cho các cột cần
        if col in zero_invalid_cols:
            invalid_mask |= (df[col] == 0)

        invalid_values = df.loc[invalid_mask, col]

        if not invalid_values.empty:
            errors[col] = {
                "count": invalid_mask.sum(),
                "min_actual": df[col].min(),
                "max_actual": df[col].max(),
                "problem_values": invalid_values.tolist()
            }

    return errors

In [10]:
errors= detect_physiological_errors(df_dataset,physiological_ranges)
print ("====Dữ liệu lỗi được phát hiện====")
for col, info in errors.items():
    print(f"{col}:{info['count']} giá trị lỗi")
    print(f"- Range thực tế: {info['min_actual']}-{info['max_actual']}")
    print(f"- Giá trị có vấn đề  {info['problem_values']}")

====Dữ liệu lỗi được phát hiện====
Glucose:5 giá trị lỗi
- Range thực tế: 0-199
- Giá trị có vấn đề  [0, 0, 0, 0, 0]
BloodPressure:35 giá trị lỗi
- Range thực tế: 0-122
- Giá trị có vấn đề  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
SkinThickness:227 giá trị lỗi
- Range thực tế: 0-99
- Giá trị có vấn đề  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

## 4. Xử lý dữ liệu & Chuẩn bị Train

In [11]:
from xu_ly_du_lieu import xulydulieu

# Xóa dòng trùng lặp
df_clean = df_dataset.drop_duplicates()
print(f'Sau khi xóa trùng lặp: {df_clean.shape}')

# Áp dụng hàm xử lý dữ liệu (thay thế 0 bằng median, lọc ngoại lệ)
df_final = xulydulieu(df_clean, physiological_ranges)
print(f'Sau khi xử lý: {df_final.shape}')

Sau khi xóa trùng lặp: (768, 9)
Sau khi xử lý: (768, 9)


## 5. Chia dữ liệu Train / Test (7/3)

In [12]:
import copy
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import pandas as pd
import os

datatemp = copy.copy(df_dataset)
X = datatemp.drop('Outcome', axis=1)
y = datatemp['Outcome']
seed = 1

# chia train/test (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed, stratify=y
)
# chia train → train_sub/validation
X_train_sub, X_valid, y_train_sub, y_valid = train_test_split(
    X_train, y_train, test_size=0.3, random_state=seed, stratify=y_train
)

print('Train size:', X_train.shape)
print('Test size: ', X_test.shape)
print('Train sub: ', X_train_sub.shape)
print('Validation:', X_valid.shape)

Train size: (537, 8)
Test size:  (231, 8)
Train sub:  (375, 8)
Validation: (162, 8)


## 7. SVM (Support Vector Machine)

In [13]:
# Khởi tạo và huấn luyện SVM
svm_model = SVC(kernel='linear', random_state=seed) # Bạn có thể đổi kernel thành 'rbf', 'poly'...
svm_model.fit(X_train_sub, y_train_sub)
y_pred_valid = svm_model.predict(X_valid)
acc_valid = accuracy_score(y_valid, y_pred_valid)
print(f'Validation Accuracy: {acc_valid:.4f}')

svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
acc_test = accuracy_score(y_test, y_pred_svm)
print(f'Test Accuracy: {acc_test:.4f}')
print(confusion_matrix(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

Validation Accuracy: 0.7840
Test Accuracy: 0.7706
[[134  16]
 [ 37  44]]
              precision    recall  f1-score   support

           0       0.78      0.89      0.83       150
           1       0.73      0.54      0.62        81

    accuracy                           0.77       231
   macro avg       0.76      0.72      0.73       231
weighted avg       0.77      0.77      0.76       231



In [16]:
# Đánh giá K-Fold
scores_svm = cross_val_score(svm_model, X_train, y_train, cv=10)
print('Scores từng fold:', scores_svm)
print(f'Mean Accuracy: {scores_svm.mean():.4f}')

Scores từng fold: [0.74074074 0.72222222 0.74074074 0.7962963  0.81481481 0.74074074
 0.74074074 0.77358491 0.8490566  0.79245283]
Mean Accuracy: 0.7711


In [17]:
# Lưu kết quả vào Excel (dữ liệu gốc)
folder_path = 'ket_qua'
file_path = os.path.join(folder_path, 'ket_qua_modelSVM.xlsx')

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_svm).ravel()
f1 = f1_score(y_test, y_pred_svm)

new_summary_df = pd.DataFrame({
    'Validation Accuracy': [acc_valid],
    'Test Accuracy':       [acc_test],
    'Mean KFold Accuracy': [scores_svm.mean()],
    'TP': [tp], 'FP': [fp], 'FN': [fn], 'TN': [tn],
    'F1-score': [f1]
})

os.makedirs(folder_path, exist_ok=True)
if os.path.exists(file_path):
    old_df = pd.read_excel(file_path, sheet_name='Summary')
    summary_df = pd.concat([old_df, new_summary_df], ignore_index=True)
else:
    summary_df = new_summary_df

with pd.ExcelWriter(file_path, engine='openpyxl', mode='w') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print('Đã ghi kết quả SVM (dữ liệu gốc) vào thư mục ket_qua')

Đã ghi kết quả SVM (dữ liệu gốc) vào thư mục ket_qua


In [18]:
X2 = df_final.drop('Outcome', axis=1)
y2 = df_final['Outcome']

# chia train/test (70/30)
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.3, random_state=seed, stratify=y2
)
# chia train → train_sub/validation
X_train_sub2, X_valid2, y_train_sub2, y_valid2 = train_test_split(
    X_train2, y_train2, test_size=0.3, random_state=seed, stratify=y_train2
)

In [19]:
# Khởi tạo và huấn luyện SVM trên dữ liệu sạch
svm_model2 = SVC(kernel='linear', random_state=seed)
svm_model2.fit(X_train_sub2, y_train_sub2)
y_pred_valid2 = svm_model2.predict(X_valid2)
acc_valid2 = accuracy_score(y_valid2, y_pred_valid2)
print(f'Validation Accuracy: {acc_valid2:.4f}')

svm_model2.fit(X_train2, y_train2)
y_pred_svm2 = svm_model2.predict(X_test2)
acc_test2 = accuracy_score(y_test2, y_pred_svm2)
print(f'Test Accuracy: {acc_test2:.4f}')
print(confusion_matrix(y_test2, y_pred_svm2))
print(classification_report(y_test2, y_pred_svm2))

Validation Accuracy: 0.7840
Test Accuracy: 0.7532
[[133  17]
 [ 40  41]]
              precision    recall  f1-score   support

           0       0.77      0.89      0.82       150
           1       0.71      0.51      0.59        81

    accuracy                           0.75       231
   macro avg       0.74      0.70      0.71       231
weighted avg       0.75      0.75      0.74       231



In [20]:
# Đánh giá K-Fold
scores_svm2 = cross_val_score(svm_model2, X_train2, y_train2, cv=10)
print('Scores từng fold:', scores_svm2)
print(f'Mean Accuracy: {scores_svm2.mean():.4f}')

Scores từng fold: [0.7037037  0.72222222 0.74074074 0.7962963  0.81481481 0.75925926
 0.74074074 0.77358491 0.83018868 0.77358491]
Mean Accuracy: 0.7655


In [21]:
# Lưu kết quả vào Excel (dữ liệu đã xử lý)
file_path2 = os.path.join(folder_path, 'ket_qua_modelSVM_processed.xlsx')

tn2, fp2, fn2, tp2 = confusion_matrix(y_test2, y_pred_svm2).ravel()
f1_2 = f1_score(y_test2, y_pred_svm2)

new_summary_df2 = pd.DataFrame({
    'Validation Accuracy': [acc_valid2],
    'Test Accuracy':       [acc_test2],
    'Mean KFold Accuracy': [scores_svm2.mean()],
    'TP': [tp2], 'FP': [fp2], 'FN': [fn2], 'TN': [tn2],
    'F1-score': [f1_2]
})

if os.path.exists(file_path2):
    old_df2 = pd.read_excel(file_path2, sheet_name='Summary')
    summary_df2 = pd.concat([old_df2, new_summary_df2], ignore_index=True)
else:
    summary_df2 = new_summary_df2

with pd.ExcelWriter(file_path2, engine='openpyxl', mode='w') as writer:
    summary_df2.to_excel(writer, sheet_name='Summary', index=False)

print('Đã ghi kết quả SVM (dữ liệu đã xử lý) vào thư mục ket_qua')

Đã ghi kết quả SVM (dữ liệu đã xử lý) vào thư mục ket_qua


##### Thay đổi các thông số mô hình

In [ ]:
svm_model3 = SVC(
    kernel='rbf',
    C=1,
    gamma='scale',
    class_weight='balanced',
    random_state=seed
)

svm_model3.fit(X_train_sub2, y_train_sub2)
y_pred_valid3 = svm_model3.predict(X_valid2)
acc_valid3 = accuracy_score(y_valid2, y_pred_valid3)

svm_model3.fit(X_train2, y_train2)
y_pred_svm3 = svm_model3.predict(X_test2)
acc_test3 = accuracy_score(y_test2, y_pred_svm3)

print('ValidAccuracy:', acc_valid3)
print('Accuracy:', acc_test3)
print(confusion_matrix(y_test2, y_pred_svm3))
for k, v in svm_model3.get_params().items():
    print(f'{k}: {v}')

##### kfold

In [ ]:
scores_svm3 = cross_val_score(svm_model3, X_train2, y_train2, cv=10)
svm_model3.fit(X_train2, y_train2)
print('Scores từng fold:', scores_svm3)
print('Mean Accuracy:', scores_svm3.mean())
for k, v in svm_model3.get_params().items():
    print(f'{k}: {v}')

In [ ]:
folder_path = 'ket_qua'
file_path3 = os.path.join(folder_path, 'ket_qua_modelSVM_processed.xlsx')

tn3, fp3, fn3, tp3 = confusion_matrix(y_test2, y_pred_svm3).ravel()
f1_3 = f1_score(y_test2, y_pred_svm3)

new_summary_df3 = pd.DataFrame({
    'Validation Accuracy': [acc_valid3],
    'Test Accuracy':       [acc_test3],
    'Mean KFold Accuracy': [scores_svm3.mean()],
    'TP': [tp3], 'FP': [fp3], 'FN': [fn3], 'TN': [tn3],
    'F1-score': [f1_3]
})

os.makedirs(folder_path, exist_ok=True)
if os.path.exists(file_path3):
    old_df3 = pd.read_excel(file_path3, sheet_name='Summary')
    summary_df3 = pd.concat([old_df3, new_summary_df3], ignore_index=True)
else:
    summary_df3 = new_summary_df3

with pd.ExcelWriter(file_path3, engine='openpyxl', mode='w') as writer:
    summary_df3.to_excel(writer, sheet_name='Summary', index=False)

print('Đã ghi thêm TP, FP, FN, TN, F1 vào Summary')

In [ ]:
!jupyter nbconvert --to html SVM.ipynb